# 02 — Baseline Experiments (TF-IDF + Linear models)

Train and inspect the TF-IDF + Logistic Regression / Linear SVM baselines, including the most informative n-grams and a first look at errors.

In [ ]:
import sys; sys.path.insert(0,'../src')
import pandas as pd, numpy as np
from political_sentiment.data.io import load_frame
from political_sentiment.models.baseline import TfidfLinearModel
from political_sentiment.evaluation.metrics import compute_metrics, confusion
from political_sentiment.evaluation.plots import plot_confusion_matrix

In [ ]:
train = load_frame('../data/processed','train'); test = load_frame('../data/processed','test')

## Logistic Regression vs Linear SVM

In [ ]:
rows=[]
for clf in ['logistic_regression','linear_svm']:
    m = TfidfLinearModel(classifier=clf).fit(train)
    proba = m.predict_proba(test['text'].tolist())
    preds = [m.labels[i] for i in proba.argmax(1)]
    met = compute_metrics(test['label'].tolist(), preds, proba, m.labels)
    met['model']=clf; rows.append(met)
pd.DataFrame(rows).set_index('model').round(4)

## Most informative n-grams (interpretability)

In [ ]:
m = TfidfLinearModel(classifier='logistic_regression').fit(train)
feats = m.top_features(12)
for lbl, items in feats.items():
    print('\n', lbl.upper()); print(', '.join(w for w,_ in items))

## Confusion matrix

In [ ]:
proba = m.predict_proba(test['text'].tolist()); preds=[m.labels[i] for i in proba.argmax(1)]
cm = confusion(test['label'].tolist(), preds, m.labels)
plot_confusion_matrix(cm, '/tmp/cm_base.png', m.labels, title='Baseline confusion'); 
from IPython.display import Image; Image('/tmp/cm_base.png')

## Hardest errors
Most-confident misclassifications — dominated by negation, contrast and sarcasm, and by cross-lingual (Bangla) negation.

In [ ]:
from political_sentiment.evaluation.error_analysis import collect_errors, error_rate_by
err = collect_errors(test, preds, proba, m.labels)
err.head(10)

In [ ]:
error_rate_by(test, preds, 'language')